# Hyperparameter Sensitivity Atlas

*Companion to* **Mastering CatBoost — Pro Edition**, Chapter 4: Tuning & Diagnostics as System Control, §4.3 (The Stability Triangle) and §4.2 (The Hierarchy of Needs).

Before you can balance the Stability Triangle you need to know how steep each edge is *on your dataset*. This notebook builds a one-at-a-time (OAT) sensitivity atlas: sweeps of `depth`, `learning_rate`, `l2_leaf_reg`, and `border_count` around a fixed baseline, each plotted with its accuracy/time trade-off — then a depth × L2 interaction map that shows exactly where OAT reasoning breaks down.

In [ ]:
%pip install -q catboost==1.2.10 scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0

import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

## 1. Baseline and development protocol

Each one-at-a-time sweep varies one parameter while the others remain fixed. We record validation AUC, fit time, retained trees, and the train-validation AUC gap. The gap is descriptive; it is not a variance estimator and also reflects fit, split sampling, stopping, and selection. This atlas explores a seeded development split and does not open a final test set.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=12_000, n_features=25, n_informative=10,
                           n_redundant=6, flip_y=0.1, random_state=RANDOM_SEED)
X = X.astype(np.float32)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y)

BASELINE = dict(depth=6, l2_leaf_reg=3.0, learning_rate=0.05, border_count=254)

def probe(**overrides):
    params = {**BASELINE, **overrides}
    m = CatBoostClassifier(iterations=600, eval_metric="AUC", random_seed=RANDOM_SEED,
                           verbose=0, allow_writing_files=False, **params)
    t0 = time.perf_counter()
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50, use_best_model=True)
    dt = time.perf_counter() - t0
    tr = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
    va = roc_auc_score(y_va, m.predict_proba(X_va)[:, 1])
    return {"valid_auc": va, "gap": tr - va, "fit_s": dt, "trees": m.tree_count_}

## 2. The four sweeps

- **`depth`** — Corner 2 of the Stability Triangle: each +1 doubles the leaves of a symmetric tree ($2^d$); the book reserves 10+ for very large data (§4.3.2).
- **`learning_rate`** — Corner 1: remember the inverse law $N \approx C/\eta$; with early stopping active, watch the `trees` column compensate.
- **`l2_leaf_reg`** — Corner 3: log-spaced sweep, since the penalty acts multiplicatively.
- **`border_count`** — Data Layer resolution (§4.1 map, Chapter 3 §3.2.2): included because it feeds both engines upstream of the Triangle.

In [ ]:
sweeps = {
    "depth":         [2, 3, 4, 6, 8, 10],
    "learning_rate": [0.005, 0.02, 0.05, 0.1, 0.2, 0.4],
    "l2_leaf_reg":   [0.2, 1.0, 3.0, 9.0, 27.0, 81.0],
    "border_count":  [8, 32, 64, 128, 254],
}

atlas = {}
for param, values in sweeps.items():
    rows = []
    for v in values:
        res = probe(**{param: v})
        rows.append({param: v, **res})
        print(f"{param}={v}: valid AUC {res['valid_auc']:.4f}, gap {res['gap']:.4f}, "
              f"{res['fit_s']:.1f}s, {res['trees']} trees")
    atlas[param] = pd.DataFrame(rows)
print("\natlas complete")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
log_axes = {"learning_rate", "l2_leaf_reg"}
for ax, (param, df) in zip(axes.ravel(), atlas.items()):
    ax.plot(df[param], df["valid_auc"], "o-", color="tab:blue", label="valid AUC")
    ax.set_xlabel(param); ax.set_ylabel("valid AUC", color="tab:blue")
    if param in log_axes:
        ax.set_xscale("log")
    ax2 = ax.twinx()
    ax2.plot(df[param], df["fit_s"], "s--", color="tab:red", alpha=0.7, label="fit time")
    ax2.set_ylabel("fit time (s)", color="tab:red")
    ax.set_title(f"OAT sweep: {param}")
plt.suptitle("Hyperparameter Sensitivity Atlas (baseline: depth=6, l2=3, lr=0.05, borders=254)")
plt.tight_layout(); plt.show()

In [ ]:
# Descriptive train-validation gap; do not label it an estimator of variance.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, (param, frame) in zip(axes, atlas.items()):
    ax.plot(frame[param], frame["gap"], "o-", color="tab:purple")
    if param in log_axes:
        ax.set_xscale("log")
    ax.set_xlabel(param); ax.set_title("train-valid AUC gap")
axes[0].set_ylabel("descriptive gap")
plt.tight_layout(); plt.show()

### Reading the atlas

Read the live plots rather than expecting prescribed shapes. Depth, learning rate, L2, and border count can interact with early stopping, noise, and the seeded split. The common heuristic that useful tree count often rises as learning rate falls is not an inverse law. Rank sensitivity only for the declared development metric and include repeated seeds before allocating substantial tuning budget.

## 3. Where one-at-a-time sweeps break: depth x L2

A small grid can reveal an interaction that separate sweeps cannot. Whether the validation-selected L2 shifts with depth is an empirical result, not a required consequence.

In [ ]:
depths = [4, 6, 8, 10]
l2s = [0.5, 3.0, 9.0, 27.0]
heat = np.zeros((len(depths), len(l2s)))
for i, d in enumerate(depths):
    for j, l2 in enumerate(l2s):
        heat[i, j] = probe(depth=d, l2_leaf_reg=l2)["valid_auc"]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(heat, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(l2s)), [str(v) for v in l2s])
ax.set_yticks(range(len(depths)), [str(v) for v in depths])
ax.set_xlabel("l2_leaf_reg"); ax.set_ylabel("depth")
for i in range(len(depths)):
    for j in range(len(l2s)):
        ax.text(j, i, f"{heat[i,j]:.4f}", ha="center", va="center",
                color="white", fontsize=9)
plt.colorbar(im, label="valid AUC")
plt.title("depth x l2_leaf_reg interaction (valid AUC)")
plt.tight_layout(); plt.show()

best_l2_per_depth = {d: l2s[int(np.argmax(heat[i]))] for i, d in enumerate(depths)}
print("best l2_leaf_reg at each depth:", best_l2_per_depth)

Inspect the printed grid and report whether the selected L2 changes with depth in this seed. Leaf occupancy motivates smoothing candidates, but no monotone shift or universal optimum follows.

## Takeaways

- One-at-a-time curves are development diagnostics and miss interactions.
- Train-validation gap is not a variance estimator.
- Parameter sensitivity rankings are dataset, metric, budget, seed, and protocol dependent.
- Sweep multiplicative ranges on a log scale when that matches the parameter geometry.
- Treat the learning-rate/tree-count relation as a budgeting heuristic, not a law.

## Exercises

**Exercise 1 - Time-normalized sensitivity.** Define the utility/cost ratio before inspection and repeat seeds. Report how the ranking changes; do not prescribe depth or learning rate as the winner.

**Exercise 2 - Atlas under noise.** Regenerate the fixture at several label-noise levels and report validation-selected depth/L2 with uncertainty. More noise need not move either optimum monotonically.

**Exercise 3 - `rsm`.** Compare the sweep before and after adding redundant/noise features. Report live slopes; feature count alone does not determine the result.

### Hidden solutions

<details><summary>Solution</summary>

**Exercise 1.** Predeclare a utility such as `(validation_auc - baseline_auc) / fit_seconds` and a minimum practically relevant AUC difference. Refit each one-at-a-time candidate across seeds, calculate utility per run, and rank parameters from paired distributions. Show raw AUC and time too because a ratio can become unstable near zero cost or gain.

</details>

<details><summary>Solution</summary>

**Exercise 2.** Regenerate each noise level, preserve one untouched evaluation split per seed, and run the same depth-by-L2 development grid. Record the validation-selected pair before scoring that pair once on the evaluation split. Plot the distribution of selected depth/L2 and evaluation AUC; do not infer a monotone shift from one seed.

</details>

<details><summary>Solution</summary>

**Exercise 3.** Append controlled blocks of redundant and independent noise features, then rerun the same `rsm` grid on identical splits. Record resolved feature count, sampled-feature fraction, best iteration, validation score, and fit time. Compare within-seed slopes and uncertainty; feature count alone does not determine the optimum.

</details>
